## Paths & Libraries

In [ ]:
from pathlib import Path
import os
import pandas as pd

In [ ]:
def find_project_root(start_path, marker="data"):
    path = Path(start_path).resolve()
    for parent in [path] + list(path.parents):
        if (parent / marker).exists():
            return parent
    raise RuntimeError("Project root not found")

PROJECT_ROOT = find_project_root(Path.cwd())
os.chdir(PROJECT_ROOT)

print("Project root:", PROJECT_ROOT)

DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
DATA_BATCHES =  PROJECT_ROOT / "data" / "batches"
DATA_OUTPUT =  PROJECT_ROOT / "data" / "output"

In [ ]:
for i in range(5):
    batch_file = DATA_OUTPUT / f"final/batch_{i}_20260508.csv"

    if batch_file.exists():
        batch_df = pd.read_csv(batch_file)

        output_file = DATA_PROCESSED / "output_llama.csv"

        batch_df.to_csv(
            output_file,
            index=False,
            mode='a',
            header=not output_file.exists()
        )

        print(f"Merged {batch_file} into {output_file}")

    else:
        print(f"Batch file {batch_file} does not exist")

In [ ]:
out= pd.read_csv(DATA_PROCESSED / "output_llama.csv")

In [ ]:
out.columns

In [ ]:
out  = out[['companyid','transcript_id','date', 'output', 'tokens']]

In [ ]:
out.columns
out.dtypes

out = out.astype({
    'companyid': 'int64',
    'transcript_id': 'int64',
    'tokens': 'int64'
})

In [ ]:
link = pd.read_csv(DATA_PROCESSED / "sp500_link_table.csv", index_col=0)

In [ ]:
link.dtypes

In [ ]:
merged = out.merge(link, on='companyid', how='left')

In [ ]:
merged

In [ ]:
import json
import re
import pandas as pd

def parse_llm_output(text):
    if pd.isna(text):
        return None
    
    text = re.sub(r"```json\s*", "", str(text))
    text = re.sub(r"```", "", text)
    text = text.replace('""', '"')
    
    stack = []
    json_blocks = []
    start = None

    for i, char in enumerate(text):
        if char == "{":
            if not stack:
                start = i
            stack.append(char)
        elif char == "}":
            if stack:
                stack.pop()
                if not stack and start is not None:
                    json_blocks.append(text[start:i+1])

    parsed = []
    for block in json_blocks:
        try:
            parsed.append(json.loads(block))
        except json.JSONDecodeError:
            pass

    return parsed if parsed else None


def flatten_first_json(parsed):
    if not parsed:
        return pd.Series()

    d = parsed[0]

    return pd.Series({
        "forward_looking_intensity": d.get("forward_looking_intensity"),
        "specificity": d.get("specificity"),
        "economic_substance": d.get("economic_substance"),
        "tone": d.get("tone"),
        "certainty": d.get("certainty"),
        "main_focus": d.get("context_summary", {}).get("main_focus"),
        "secondary_focus": d.get("context_summary", {}).get("secondary_focus"),
        "managerial_horizon": d.get("context_summary", {}).get("managerial_horizon"),
        "overall_outlook": d.get("context_summary", {}).get("overall_outlook"),
    })


parsed_cols = merged["output"].apply(parse_llm_output).apply(flatten_first_json)

output = pd.concat([merged, parsed_cols], axis=1)

In [ ]:
output.drop(columns=["output"], inplace=True)

In [ ]:
output.columns

In [ ]:
controls = pd.read_csv(DATA_PROCESSED / "controls.csv")
controls.columns



In [ ]:
controls = controls[['transcriptid', 'companyid','companyname','date', 'log_prev_market_cap', 'public_date', 'bm', 'roa', 'roe']].copy()


In [ ]:
controls.rename(columns={'transcriptid': 'transcript_id'}, inplace=True)

In [ ]:
output.merge(controls, on=["companyid", "transcript_id", "date"], how="left")

In [ ]:
output[output.duplicated(subset=["transcript_id", "date"])]

In [ ]:
output.to_csv(DATA_PROCESSED / "output_llama_parsed.csv", index=False)